# Context-conditioned tokenization from BEATs

**Pipeline (test):** segment audio → **BEATs-pitch** (768-D) → **file-level context** (predicted) → **UMAP + HDBSCAN + NCA** tokenizer fitted on **train segments of that context only** → token id.

**Oracle:** same, but context comes from metadata (true `context_name` per segment).

**Classifier:** order-free pooling over segments in a file (**mean** and **std** of BEATs along the segment axis → 1536-D), aligned with Assom’s associative-syntax view (order not required for context v1).

**Baselines in plots:** context accuracy; **ARI** between token ids (pred vs oracle) on test segments; optional global tokenizer (one UMAP+HDBSCAN on all train segments).


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
import os
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, adjusted_rand_score, confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier, NeighborhoodComponentsAnalysis
from sklearn.pipeline import Pipeline, make_pipeline
import umap
import hdbscan

sys.path.insert(0, os.path.abspath('..'))

from src.data import load_annotations, load_segments, dynamic_segmentation, iqr_filter
from src.features import compute_beats_embeddings

print('Imports OK')


In [ ]:
# ── Configuration (align with notebooks/beats_experiment.ipynb) ──

DATA_DIR = Path('/Volumes/T7/data/raw/fruitbat')
BEATS_CKPT = '/Volumes/T7/models/beats/BEATs_iter3_plus_AS2M.pt'
SR = 250_000
RANDOM_STATE = 42
TEST_SIZE = 0.2
BEATS_BATCH = 4

USE_DYNAMIC_SEG = True
DYN_SEG_PARAMS = dict(
    n_fft=1024, hop_length_ms=0.5, win_length_ms=4, ref_level_db=20,
    pre=0.97, min_level_db=-30, silence_threshold=0.1,
    min_silence_for_spec=0.1, max_vocal_for_spec=1.0,
    min_syllable_length_s=0.01, spectral_range=[5000, 60000],
    min_level_db_floor=20, verbose=False,
)

UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.3
HDBSCAN_MIN_CLUSTER_FRAC = 0.02
HDBSCAN_MIN_SAMPLES = 20
HDBSCAN_EPSILON = 0.1
NCA_KNN_NEIGHBORS = 30

# Per-context tokenizer: skip fitting dedicated UMAP if fewer than this many train segments
MIN_SEGMENTS_PER_CONTEXT = 500

print('Config OK')


## 1. Load segments

In [ ]:
df = load_annotations(DATA_DIR)
seg_df = load_segments(df, DATA_DIR)
if USE_DYNAMIC_SEG:
    seg_df = dynamic_segmentation(seg_df, DYN_SEG_PARAMS)
seg_df = iqr_filter(seg_df)
seg_df = seg_df.reset_index(drop=True)
print(len(seg_df), 'segments')
print(seg_df['context_name'].value_counts())


## 2. BEATs-pitch embeddings (768-D)

In [ ]:
%%time
X = compute_beats_embeddings(
    seg_df, checkpoint_path=BEATS_CKPT,
    mode='pitch_shift', native_sr=SR, batch_size=BEATS_BATCH,
)
print('X shape', X.shape)


## 3. File-level table, train/test split, context classifier

Pooling: **mean** and **std** over segment embeddings per `file_name` (order-free). Label = **dominant** `context_name` in file (mode).


In [ ]:
file_rows = []
for fname, g in seg_df.groupby('file_name', sort=False):
    file_rows.append({
        'file_name': fname,
        'context_name': g['context_name'].mode().iloc[0],
        'n_seg': len(g),
    })
file_meta = pd.DataFrame(file_rows)

# Build [mean || std || log(n+1)] per file (order-free over segments)
rows = []
for fname in file_meta['file_name']:
    idx = seg_df.index[seg_df['file_name'] == fname].to_numpy()
    block = X[idx]
    mu = block.mean(axis=0)
    sig = block.std(axis=0)
    feat = np.concatenate([mu, sig, [np.log(len(idx) + 1)]])
    rows.append(feat)

feat_by_file = dict(zip(file_meta['file_name'], rows))
y_file_ctx = file_meta['context_name'].values

le_ctx = LabelEncoder()
y_enc = le_ctx.fit_transform(y_file_ctx)

f_train, f_test, y_train, y_test = train_test_split(
    file_meta['file_name'].values,
    y_enc,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_enc,
)

train_files = set(f_train)
test_files = set(f_test)
train_seg_mask = seg_df['file_name'].isin(train_files).values
test_seg_mask = seg_df['file_name'].isin(test_files).values

Xf_train = np.stack([feat_by_file[f] for f in f_train])
Xf_test = np.stack([feat_by_file[f] for f in f_test])

scaler_f = StandardScaler()
Xf_train_s = scaler_f.fit_transform(Xf_train)
Xf_test_s = scaler_f.transform(Xf_test)

clf_ctx = SVC(kernel='rbf', C=10, gamma='scale', random_state=RANDOM_STATE)
clf_ctx.fit(Xf_train_s, y_train)
y_pred_file = clf_ctx.predict(Xf_test_s)

acc_ctx = accuracy_score(y_test, y_pred_file)
f1_ctx = f1_score(y_test, y_pred_file, average='macro')
print(f'Hold-out file context — accuracy: {acc_ctx:.3f}, macro-F1: {f1_ctx:.3f}')

# CV on train files (Scaler inside each fold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
clf_cv = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', C=10, gamma='scale', random_state=RANDOM_STATE),
)
y_cv = cross_val_predict(clf_cv, Xf_train, y_train, cv=cv)
print('5-fold CV on train files — acc:', accuracy_score(y_train, y_cv), 'macro-F1:', f1_score(y_train, y_cv, average='macro'))

# Map file -> predicted context name (test only)
file_to_pred_ctx = {fn: le_ctx.inverse_transform([yp])[0] for fn, yp in zip(f_test, y_pred_file)}
file_to_true_ctx_mode = {fn: le_ctx.inverse_transform([yt])[0] for fn, yt in zip(f_test, y_test)}


## 4. Baseline helpers: NCA noise reassignment + fit bundle

In [ ]:
def reassign_noise_nca(labels: np.ndarray, emb: np.ndarray, n_neighbors: int = NCA_KNN_NEIGHBORS) -> np.ndarray:
    labels_ext = labels.copy()
    ix_good = np.where(labels >= 0)[0]
    ix_noise = np.where(labels == -1)[0]
    if len(ix_noise) == 0 or len(ix_good) < 10:
        return labels_ext
    try:
        pipe = Pipeline([
            ('nca', NeighborhoodComponentsAnalysis(random_state=RANDOM_STATE)),
            ('knn', KNeighborsClassifier(n_neighbors=n_neighbors, n_jobs=-1)),
        ])
        pipe.fit(emb[ix_good], labels[ix_good])
        labels_ext[ix_noise] = pipe.predict(emb[ix_noise])
    except Exception:
        knn = KNeighborsClassifier(n_neighbors=n_neighbors, n_jobs=-1)
        knn.fit(emb[ix_good], labels[ix_good])
        labels_ext[ix_noise] = knn.predict(emb[ix_noise])
    return labels_ext


@dataclass
class ContextTokenizerBundle:
    context_name: str
    reducer: object
    clusterer: object
    emb2d_train: np.ndarray
    labels_train: np.ndarray
    knn_fallback: KNeighborsClassifier


def fit_tokenizer_bundle(context_name: str, X_sub: np.ndarray) -> Optional[ContextTokenizerBundle]:
    N = len(X_sub)
    if N < 30:
        return None
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric='euclidean',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    emb2d = reducer.fit_transform(X_sub)
    mcs = max(int(N * HDBSCAN_MIN_CLUSTER_FRAC), 10)
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        min_samples=HDBSCAN_MIN_SAMPLES,
        cluster_selection_epsilon=HDBSCAN_EPSILON,
        cluster_selection_method='leaf',
        prediction_data=True,
    )
    clusterer.fit(emb2d)
    raw = clusterer.labels_.copy()
    labels = reassign_noise_nca(raw, emb2d)
    knn = KNeighborsClassifier(n_neighbors=min(NCA_KNN_NEIGHBORS, max(1, N - 1)), weights='uniform', n_jobs=-1)
    knn.fit(emb2d, labels)
    return ContextTokenizerBundle(context_name, reducer, clusterer, emb2d, labels, knn)


def predict_token(b: ContextTokenizerBundle, x_768: np.ndarray) -> int:
    z = b.reducer.transform(x_768.reshape(1, -1))
    lbl, _ = hdbscan.approximate_predict(b.clusterer, z)
    if int(lbl[0]) < 0:
        return int(b.knn_fallback.predict(z)[0])
    return int(lbl[0])


def predict_tokens_batch(b: ContextTokenizerBundle, X_sub: np.ndarray) -> np.ndarray:
    z = b.reducer.transform(X_sub)
    lbl, _ = hdbscan.approximate_predict(b.clusterer, z)
    out = lbl.astype(int).copy()
    noise = out < 0
    if noise.any():
        out[noise] = b.knn_fallback.predict(z[noise])
    return out


## 5. Fit per-context tokenizers (train) + global fallback

In [ ]:
X_train = X[train_seg_mask]
ctx_train = seg_df.loc[train_seg_mask, 'context_name'].values

global_bundle = fit_tokenizer_bundle('__global__', X_train)
assert global_bundle is not None

tokenizers: dict[str, ContextTokenizerBundle] = {}
for ctx in sorted(pd.Series(ctx_train).unique()):
    m = ctx_train == ctx
    n = m.sum()
    if n < MIN_SEGMENTS_PER_CONTEXT:
        print(f'Skip dedicated tokenizer for {ctx!r} (n={n} < {MIN_SEGMENTS_PER_CONTEXT}); use global')
        continue
    b = fit_tokenizer_bundle(ctx, X_train[m])
    if b is None:
        print(f'Fit failed for {ctx!r}')
        continue
    tokenizers[ctx] = b
    ncl = len(np.unique(b.labels_train))
    print(f'Tokenizer {ctx!r}: train N={n}, clusters~{ncl}')

print('Dedicated contexts:', list(tokenizers.keys()))


## 6. Inference on test segments (pred vs oracle)

In [ ]:
def bundle_for_context(ctx_name: str) -> ContextTokenizerBundle:
    return tokenizers.get(ctx_name, global_bundle)


# file -> dominant true context (same as classifier target, for oracle-by-file variant)
file_to_true_mode = dict(zip(file_meta['file_name'], file_meta['context_name']))

y_tok_pred = np.zeros(test_seg_mask.sum(), dtype=int)
y_tok_oracle = np.zeros_like(y_tok_pred)
y_tok_oracle_seg = np.zeros_like(y_tok_pred)

test_indices = np.where(test_seg_mask)[0]
for j, i in enumerate(test_indices):
    fn = seg_df.at[i, 'file_name']
    ctx_hat = file_to_pred_ctx[fn]
    ctx_true_mode = file_to_true_mode[fn]
    ctx_true_seg = seg_df.at[i, 'context_name']

    y_tok_pred[j] = predict_token(bundle_for_context(ctx_hat), X[i])
    y_tok_oracle[j] = predict_token(bundle_for_context(ctx_true_mode), X[i])
    y_tok_oracle_seg[j] = predict_token(bundle_for_context(ctx_true_seg), X[i])

ari_pred_vs_oracle_mode = adjusted_rand_score(y_tok_oracle, y_tok_pred)
ari_pred_vs_oracle_seg = adjusted_rand_score(y_tok_oracle_seg, y_tok_pred)
print('ARI tokens: pred vs oracle (file mode context):', ari_pred_vs_oracle_mode)
print('ARI tokens: pred vs oracle (segment context):', ari_pred_vs_oracle_seg)

# Global tokenizer on same test points (reference)
glob_test = predict_tokens_batch(global_bundle, X[test_indices])
ari_pred_vs_global = adjusted_rand_score(glob_test, y_tok_pred)
print('ARI tokens: pred pipeline vs global-only labels:', ari_pred_vs_global)


## 7. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred_file)
disp = ConfusionMatrixDisplay(cm, display_labels=le_ctx.classes_)
disp.plot(ax=ax, xticks_rotation=45, values_format='d')
ax.set_title('Context classifier (hold-out files)\nrows=true, cols=pred')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
metrics = {
    'acc_context': acc_ctx,
    'ARI_pred_vs_oracle': ari_pred_vs_oracle_mode,
}
ax.bar(list(metrics.keys()), list(metrics.values()), color=['steelblue', 'darkseagreen'])
ax.set_ylim(0, 1)
ax.set_title('Summary metrics (test files / test segments)')
for i, (k, v) in enumerate(metrics.items()):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.show()

# UMAP plots for up to 3 contexts with dedicated tokenizers
show_ctx = [c for c in ['Feeding', 'Fighting', 'Isolation'] if c in tokenizers][:3]
if not show_ctx:
    show_ctx = list(tokenizers.keys())[:3]
if not show_ctx:
    print('No dedicated per-context tokenizers to plot.')
else:
    fig, axes = plt.subplots(1, len(show_ctx), figsize=(5 * len(show_ctx), 4))
    if len(show_ctx) == 1:
        axes = [axes]
    for ax, ctx in zip(axes, show_ctx):
        b = tokenizers[ctx]
        ax.scatter(b.emb2d_train[:, 0], b.emb2d_train[:, 1], c=b.labels_train, s=4, cmap='tab20', alpha=0.5)
        ax.set_title(f'Train 2D UMAP — {ctx}\n(NCA cluster id color)')
        ax.set_xlabel('UMAP-1')
        ax.set_ylabel('UMAP-2')
    plt.tight_layout()
    plt.show()


## Notes / risks

- Rare contexts may use the **global** tokenizer only.
- **UMAP.transform** on out-of-distribution segments can be noisy; KNN fallback mitigates HDBSCAN `-1`.
- Context errors directly change which tokenizer runs — measured by ARI(pred vs oracle).
